In [ ]:
# =============================================================================
# Cell 1: Configuration
# =============================================================================
# NOTEBOOK: Extract and Georeference VCF Metrics for Comparison
# =============================================================================
# Purpose: Extract specific metrics from PACE and MODIS, properly 
#          georeference them, and save as individual GeoTIFFs
#
# Now includes PACE alternative sorting metrics for comparison
#
# Output: Individual .tif files per tile × source × metric
#
# Author: MJ Frost
# Date: July 2026
# =============================================================================

from pathlib import Path
import numpy as np
from osgeo import gdal, osr
from typing import Dict, List, Tuple, Optional
import logging

gdal.UseExceptions()

# =============================================================================
# CONFIGURATION - EDIT THESE VALUES AS NEEDED
# =============================================================================

# Input paths
PACE_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/output")
MODIS_BASE = Path("/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C")

# Output directory for extracted bands
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/comparison_bands")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Years
PACE_YEAR = 2025
MODIS_YEAR = 2019

# Tiles to process
TILES = [    "h08v04", "h08v05", "h09v04", "h09v05", "h10v04", "h10v05", "h10v06",
    "h11v02", "h11v03", "h11v04", "h11v05", "h11v08", "h11v09", "h11v10",
    "h12v01", "h12v02", "h12v03", "h12v04", "h12v05", "h12v09", "h12v10", 
    "h12v12", "h13v01", "h13v02", "h13v10", "h13v11", "h13v12", "h16v01", 
    "h17v05", "h18v03", "h18v04", "h18v07", "h19v04", "h19v07", "h19v08", 
    "h19v09", "h19v10", "h19v11", "h19v12", "h20v02", "h20v03", "h20v04", 
    "h20v06", "h20v08", "h20v09", "h20v10", "h20v11", "h21v01", "h21v02",
    "h21v04", "h21v05", "h21v06", "h21v10", "h22v03", "h22v04", "h23v02", 
    "h23v03", "h24v02", "h24v03", "h24v04", "h26v06", "h27v04", "h27v06", 
    "h27v07", "h28v11", "h29v11", "h29v12", "h30v12", "h31v11"
]

# =============================================================================
# METRICS TO EXTRACT
# =============================================================================

# Original MODIS-compatible metrics (available in both PACE and MODIS)
MODIS_METRICS = [
    'Lowest6MeanBandRefl-Band_1',
    'Greenest6MeanBandRefl-Band_7', 
    'BandReflMedian-NDVI',
    'TempMeanWarmest3',

    # Matches the SECTION 6 comparison set in 4f_metrics_validation_thermal.ipynb
    'BandReflMax-NDVI',
    'BandReflMin-NDVI',
    'BandReflMedian-Band_1',
    'BandReflMedian-Band_2',
    'BandReflMedian-Band_6',
    'Greenest3MeanBandRefl-NDVI',
]

# PACE-only standard metrics (from PACE_Metrics.tif)
PACE_STANDARD_METRICS = [
    'PACEIndex-EVI-Median',
    'PACEIndex-PRI-Median',
    'PACEIndex-mARI-Median',
    'PACEIndex-CCI-Median',
    'PACEIndex-MTCI-Median',
    'PACEIndex-NDWI-Median',
]

# PACE alternative sorting metrics (from PACE_AltSort_Metrics.tif) - NEW
PACE_ALTSORT_METRICS = [
    # Brownest (senescence/dormant)
    'PACEIndex-EVI-Brownest3Mean',
    'PACEIndex-mARI-Brownest3Mean',
    'NDVI-Brownest3Mean',
    
    # Anthocyanin dynamics (mARI-sorted)
    'PACEIndex-EVI-HighAnthocyanin3Mean',
    'PACEIndex-EVI-LowAnthocyanin3Mean',
    'PACEIndex-CCI-AmpAnthocyanin',
    
    # Stress dynamics (PRI-sorted)
    'PACEIndex-EVI-LeastStressed3Mean',
    'PACEIndex-EVI-MostStressed3Mean',
    'PACEIndex-mARI-AmpStress',
    
    # Pigment dynamics (CCI-sorted)
    'PACEIndex-EVI-HighChlorophyll3Mean',
    'PACEIndex-EVI-HighCarotenoid3Mean',
    'PACEIndex-MTCI-AmpPigmentRatio',
    
    # Phenology
    'Phenology-MaxGreenupRate',
    'Phenology-MaxSenescenceRate',
    'Phenology-GrowingSeasonLength',
    'Phenology-TimeToPeak',
    'Phenology-SeasonalAsymmetry',
    
    # Cross-index relationships
    'CrossIndex-MTCI-EVI-Ratio',
    'CrossIndex-PRI-SeasonalRange',
    'CrossIndex-mARI-SeasonalRange',
    'CrossIndex-NDWI-DrySeason',
    'CrossIndex-NDWI-WetSeason',

    # =========================================================================
    # Enhanced Phenological Sorting Metrics
    # (previously a separate, never-assigned list literal after this point --
    # these ~40 metrics were silently never extracted/validated despite
    # looking like part of PACE_ALTSORT_METRICS; now actually included)
    # =========================================================================
    
    # Green-up sorted (values at spring/wet season onset)
    'Phenology-GreenUpRate',
    'PACEIndex-EVI-AtGreenUp',
    'PACEIndex-CCI-AtGreenUp',
    'PACEIndex-CIRE-AtGreenUp',
    'PACEIndex-NDWI-AtGreenUp',
    'PACEIndex-PRI-AtGreenUp',
    'NDVI-AtGreenUp',
    'NDVI-BeforeGreenUp',
    
    # Senescence sorted (values at fall/dry season)
    'Phenology-SenescenceRate',
    'PACEIndex-EVI-BeforeSenescence',
    'PACEIndex-EVI-AfterSenescence',
    'PACEIndex-EVI-SenescenceChange',
    'PACEIndex-CCI-BeforeSenescence',
    'PACEIndex-CCI-AfterSenescence',
    'PACEIndex-CCI-SenescenceChange',
    'PACEIndex-CIRE-BeforeSenescence',
    'PACEIndex-CIRE-SenescenceChange',
    'NDVI-BeforeSenescence',
    'NDVI-AfterSenescence',
    
    # Peak-NDVI sorted (values at maximum greenness)
    'PACEIndex-EVI-AtPeakNDVI',
    'PACEIndex-CCI-AtPeakNDVI',
    'PACEIndex-CIRE-AtPeakNDVI',
    'PACEIndex-NDWI-AtPeakNDVI',
    'PACEIndex-PRI-AtPeakNDVI',
    'PACEIndex-mARI-AtPeakNDVI',
    
    # Dormancy sorted (values at minimum greenness)
    'PACEIndex-EVI-AtMinNDVI',
    'PACEIndex-CCI-AtMinNDVI',
    'PACEIndex-CIRE-AtMinNDVI',
    'PACEIndex-NDWI-AtMinNDVI',
    'PACEIndex-PRI-AtMinNDVI',
    
    # Seasonal range (amplitude of each index)
    'PACEIndex-EVI-SeasonalRange',
    'PACEIndex-CCI-SeasonalRange',
    'PACEIndex-CIRE-SeasonalRange',
    'PACEIndex-NDWI-SeasonalRange',
    'PACEIndex-PRI-SeasonalRange',
    'PACEIndex-mARI-SeasonalRange',
    'PACEIndex-NDRE-SeasonalRange',
    'PACEIndex-GCI-SeasonalRange',
    'PACEIndex-NDII-SeasonalRange',
    'PACEIndex-MTCI-SeasonalRange',
]

print(f"PACE alt-sort metrics to extract: {len(PACE_ALTSORT_METRICS)}")

# Grid sizes
PACE_TILE_SIZE = 600    # 2km resolution
MODIS_TILE_SIZE = 4800  # 250m resolution

# No-data values
NO_DATA = -10001

# Sinusoidal projection parameters
MODIS_SPHERE_RADIUS = 6371007.181
MODIS_TILE_SIZE_M = 1111950.5196666666
MODIS_UPPER_LEFT_X = -20015109.354
MODIS_UPPER_LEFT_Y = 10007554.677

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Configuration loaded")
print(f"  Tiles: {TILES}")
print(f"  MODIS-compatible metrics: {len(MODIS_METRICS)}")
print(f"  PACE standard metrics: {len(PACE_STANDARD_METRICS)}")
print(f"  PACE alt-sort metrics: {len(PACE_ALTSORT_METRICS)}")
print(f"  Output: {OUTPUT_DIR}")


# =============================================================================
# Cell 2: Sinusoidal Projection Definition
# =============================================================================

def get_sinusoidal_wkt() -> str:
    """Return the WKT for MODIS Sinusoidal projection."""
    srs = osr.SpatialReference()
    srs.ImportFromProj4(
        f"+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R={MODIS_SPHERE_RADIUS} +units=m +no_defs"
    )
    return srs.ExportToWkt()


def get_tile_bounds(tile: str) -> Tuple[float, float, float, float]:
    """Get tile bounds in sinusoidal coordinates."""
    h = int(tile[1:3])
    v = int(tile[4:6])
    
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_x = min_x + MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    min_y = max_y - MODIS_TILE_SIZE_M
    
    return (min_x, min_y, max_x, max_y)


def get_geotransform(tile: str, pixel_size: float) -> Tuple[float, float, float, float, float, float]:
    """Get GDAL GeoTransform for a tile."""
    min_x, min_y, max_x, max_y = get_tile_bounds(tile)
    return (min_x, pixel_size, 0, max_y, 0, -pixel_size)


print("Projection functions defined")


# =============================================================================
# Cell 3: Band Extraction Functions
# =============================================================================

def find_band_index(ds: gdal.Dataset, band_name: str) -> Optional[int]:
    """Find the band index (1-based) for a given band name."""
    for i in range(1, ds.RasterCount + 1):
        band = ds.GetRasterBand(i)
        desc = band.GetDescription()
        metadata = band.GetMetadata()
        
        if desc == band_name:
            return i
        if metadata.get('name') == band_name:
            return i
            
    return None


def extract_pace_band(tile: str, metric_name: str, source_file: str = "MODIS_Metrics.tif") -> Optional[np.ndarray]:
    """
    Extract a specific metric band from PACE metrics files.
    
    Parameters
    ----------
    tile : str
        Tile ID (e.g., 'h12v09')
    metric_name : str
        Name of the metric to extract
    source_file : str
        Which file to read from: 'MODIS_Metrics.tif', 'PACE_Metrics.tif', 
        or 'PACE_AltSort_Metrics.tif'
    
    Returns
    -------
    np.ndarray or None
        2D array of metric values, or None if extraction failed
    """
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / source_file
    
    if not filepath.exists():
        logger.warning(f"PACE file not found: {filepath}")
        return None
    
    ds = gdal.Open(str(filepath))
    if ds is None:
        logger.warning(f"Could not open: {filepath}")
        return None
    
    band_idx = find_band_index(ds, metric_name)
    
    if band_idx is None:
        # Try partial match
        for i in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(i)
            desc = band.GetDescription()
            if metric_name in desc or desc in metric_name:
                band_idx = i
                logger.info(f"  Found partial match: '{desc}' for '{metric_name}'")
                break
    
    if band_idx is None:
        logger.warning(f"Band '{metric_name}' not found in {filepath.name}")
        ds = None
        return None
    
    data = ds.GetRasterBand(band_idx).ReadAsArray()
    ds = None
    
    return data


def extract_modis_band(tile: str, metric_name: str) -> Optional[np.ndarray]:
    """Extract a specific metric band from MODIS metrics files."""
    
    if '-' in metric_name:
        parts = metric_name.rsplit('-', 1)
        base_metric = parts[0]
        band_suffix = parts[1]
    else:
        base_metric = metric_name
        band_suffix = None
    
    filepath = MODIS_BASE / tile / str(MODIS_YEAR) / "3-Metrics" / f"{base_metric}.tif"
    
    if not filepath.exists():
        logger.warning(f"MODIS file not found: {filepath}")
        return None
    
    ds = gdal.Open(str(filepath))
    if ds is None:
        logger.warning(f"Could not open: {filepath}")
        return None
    
    band_idx = 1
    
    if band_suffix:
        band_map = {
            'Band_1': 1, 'Band_2': 2, 'Band_3': 3, 'Band_4': 4,
            'Band_5': 5, 'Band_6': 6, 'Band_7': 7, 'NDVI': 8
        }
        band_idx = band_map.get(band_suffix, 1)
        
        if band_idx > ds.RasterCount:
            logger.warning(f"Band index {band_idx} exceeds raster count {ds.RasterCount}")
            band_idx = 1
    
    logger.info(f"  Reading {filepath.name}, band {band_idx}")
    
    data = ds.GetRasterBand(band_idx).ReadAsArray()
    ds = None
    
    return data


print("Band extraction functions defined")


# =============================================================================
# Cell 4: GeoTIFF Writing Function
# =============================================================================

def write_georeferenced_tif(
    data: np.ndarray,
    output_path: Path,
    tile: str,
    no_data: int = NO_DATA,
    description: str = ""
) -> bool:
    """Write a properly georeferenced GeoTIFF in MODIS Sinusoidal projection."""
    rows, cols = data.shape
    pixel_size = MODIS_TILE_SIZE_M / cols
    
    geotransform = get_geotransform(tile, pixel_size)
    srs_wkt = get_sinusoidal_wkt()
    
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(
        str(output_path),
        cols, rows, 1,
        gdal.GDT_Int16,
        options=['COMPRESS=LZW', 'TILED=YES']
    )
    
    if out_ds is None:
        logger.error(f"Could not create: {output_path}")
        return False
    
    out_ds.SetProjection(srs_wkt)
    out_ds.SetGeoTransform(geotransform)
    
    band = out_ds.GetRasterBand(1)
    band.WriteArray(data.astype(np.int16))
    band.SetNoDataValue(no_data)
    band.SetDescription(description)
    band.FlushCache()
    
    out_ds = None
    
    logger.info(f"  Wrote: {output_path.name}")
    return True


print("GeoTIFF writing function defined")


# =============================================================================
# Cell 5: List Available Bands (Diagnostic)
# =============================================================================

def list_pace_bands(tile: str, source_file: str = "MODIS_Metrics.tif") -> List[str]:
    """List all available band names in a PACE metrics file."""
    filepath = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / source_file
    
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return []
    
    ds = gdal.Open(str(filepath))
    bands = []
    
    for i in range(1, ds.RasterCount + 1):
        band = ds.GetRasterBand(i)
        desc = band.GetDescription()
        bands.append(desc)
    
    ds = None
    return bands


# List bands from all three PACE files
print(f"\n{'='*70}")
print(f"AVAILABLE PACE BANDS IN {TILES[0]}")
print(f"{'='*70}")

for source_file in ["MODIS_Metrics.tif", "PACE_Metrics.tif", "PACE_AltSort_Metrics.tif"]:
    filepath = PACE_BASE / TILES[0] / str(PACE_YEAR) / "3-Metrics" / source_file
    if filepath.exists():
        bands = list_pace_bands(TILES[0], source_file)
        print(f"\n{source_file}: {len(bands)} bands")
        
        # Show first 10 bands
        for b in bands[:10]:
            print(f"    {b}")
        if len(bands) > 10:
            print(f"    ... and {len(bands) - 10} more")
    else:
        print(f"\n{source_file}: NOT FOUND")


# =============================================================================
# Cell 6: Verify Target Metrics Exist
# =============================================================================

print(f"\n{'='*70}")
print("VERIFYING TARGET METRICS")
print(f"{'='*70}")

# Check MODIS-compatible metrics
modis_bands = list_pace_bands(TILES[0], "MODIS_Metrics.tif")
print("\nMODIS-compatible metrics (in MODIS_Metrics.tif):")
for metric in MODIS_METRICS:
    found = metric in modis_bands
    print(f"  {'✓' if found else '✗'} {metric}")

# Check PACE standard metrics
pace_bands = list_pace_bands(TILES[0], "PACE_Metrics.tif")
print("\nPACE standard metrics (in PACE_Metrics.tif):")
for metric in PACE_STANDARD_METRICS:
    found = metric in pace_bands
    print(f"  {'✓' if found else '✗'} {metric}")

# Check PACE alt-sort metrics
altsort_bands = list_pace_bands(TILES[0], "PACE_AltSort_Metrics.tif")
print("\nPACE alt-sort metrics (in PACE_AltSort_Metrics.tif):")
for metric in PACE_ALTSORT_METRICS:
    found = metric in altsort_bands
    status = '✓' if found else '✗'
    print(f"  {status} {metric}")


# =============================================================================
# Cell 7: Main Processing Loop
# =============================================================================

def process_all():
    """
    Extract and georeference all specified metrics for all tiles.
    
    Processes three categories:
    1. MODIS-compatible metrics (PACE vs MODIS comparison)
    2. PACE standard indices (PACE only)
    3. PACE alternative sorting metrics (PACE only)
    """
    
    print("="*70)
    print("EXTRACTING AND GEOREFERENCING METRICS")
    print("="*70)
    print(f"Tiles: {TILES}")
    print(f"MODIS-compatible metrics: {len(MODIS_METRICS)}")
    print(f"PACE standard metrics: {len(PACE_STANDARD_METRICS)}")
    print(f"PACE alt-sort metrics: {len(PACE_ALTSORT_METRICS)}")
    print("="*70)
    
    results = {'success': [], 'failed': []}
    
    for tile in TILES:
        print(f"\n{'='*60}")
        print(f"Processing tile: {tile}")
        print(f"{'='*60}")
        
        # =====================================================================
        # 1. MODIS-COMPATIBLE METRICS (PACE vs MODIS)
        # =====================================================================
        print(f"\n  --- MODIS-Compatible Metrics ---")
        
        for metric_name in MODIS_METRICS:
            print(f"\n  Metric: {metric_name}")
            
            # PACE extraction
            print(f"    Extracting PACE...")
            pace_data = extract_pace_band(tile, metric_name, "MODIS_Metrics.tif")
            
            if pace_data is not None:
                pace_output = OUTPUT_DIR / f"PACE_{PACE_YEAR}_{tile}_{metric_name}.tif"
                success = write_georeferenced_tif(
                    pace_data, pace_output, tile,
                    description=f"PACE {tile} {metric_name} {PACE_YEAR}"
                )
                if success:
                    results['success'].append(str(pace_output.name))
                else:
                    results['failed'].append(f"PACE_{tile}_{metric_name}")
            else:
                results['failed'].append(f"PACE_{tile}_{metric_name}")
            
            # MODIS extraction
            print(f"    Extracting MODIS...")
            modis_data = extract_modis_band(tile, metric_name)
            
            if modis_data is not None:
                modis_output = OUTPUT_DIR / f"MODIS_{MODIS_YEAR}_{tile}_{metric_name}.tif"
                success = write_georeferenced_tif(
                    modis_data, modis_output, tile,
                    description=f"MODIS {tile} {metric_name} {MODIS_YEAR}"
                )
                if success:
                    results['success'].append(str(modis_output.name))
                else:
                    results['failed'].append(f"MODIS_{tile}_{metric_name}")
            else:
                results['failed'].append(f"MODIS_{tile}_{metric_name}")
        
        # =====================================================================
        # 2. PACE STANDARD METRICS (PACE only)
        # =====================================================================
        print(f"\n  --- PACE Standard Metrics ---")
        
        for metric_name in PACE_STANDARD_METRICS:
            print(f"\n  Metric: {metric_name}")
            print(f"    Extracting PACE...")
            
            pace_data = extract_pace_band(tile, metric_name, "PACE_Metrics.tif")
            
            if pace_data is not None:
                pace_output = OUTPUT_DIR / f"PACE_{PACE_YEAR}_{tile}_{metric_name}.tif"
                success = write_georeferenced_tif(
                    pace_data, pace_output, tile,
                    description=f"PACE {tile} {metric_name} {PACE_YEAR}"
                )
                if success:
                    results['success'].append(str(pace_output.name))
                else:
                    results['failed'].append(f"PACE_{tile}_{metric_name}")
            else:
                results['failed'].append(f"PACE_{tile}_{metric_name}")
        
        # =====================================================================
        # 3. PACE ALTERNATIVE SORTING METRICS (PACE only)
        # =====================================================================
        print(f"\n  --- PACE Alt-Sort Metrics ---")
        
        for metric_name in PACE_ALTSORT_METRICS:
            print(f"\n  Metric: {metric_name}")
            print(f"    Extracting PACE...")
            
            pace_data = extract_pace_band(tile, metric_name, "PACE_AltSort_Metrics.tif")
            
            if pace_data is not None:
                pace_output = OUTPUT_DIR / f"PACE_{PACE_YEAR}_{tile}_{metric_name}.tif"
                success = write_georeferenced_tif(
                    pace_data, pace_output, tile,
                    description=f"PACE {tile} {metric_name} {PACE_YEAR}"
                )
                if success:
                    results['success'].append(str(pace_output.name))
                else:
                    results['failed'].append(f"PACE_{tile}_{metric_name}")
            else:
                results['failed'].append(f"PACE_{tile}_{metric_name}")
    
    # Summary
    print("\n" + "="*70)
    print("PROCESSING SUMMARY")
    print("="*70)
    print(f"Successfully created: {len(results['success'])} files")
    print(f"Failed: {len(results['failed'])} files")
    
    if results['failed']:
        print("\nFailed extractions:")
        for f in results['failed']:
            print(f"  - {f}")
    
    return results


# Run the processing
results = process_all()


# =============================================================================
# Cell 8: Verify Output Files
# =============================================================================

print("="*70)
print("OUTPUT VERIFICATION")
print("="*70)

tif_files = sorted(OUTPUT_DIR.glob("*.tif"))
print(f"\nTotal files: {len(tif_files)}")

# Group by category
modis_compat_files = [f for f in tif_files if any(m in f.name for m in MODIS_METRICS)]
pace_standard_files = [f for f in tif_files if any(m in f.name for m in PACE_STANDARD_METRICS)]
pace_altsort_files = [f for f in tif_files if any(m in f.name for m in PACE_ALTSORT_METRICS)]

print(f"\nBy category:")
print(f"  MODIS-compatible: {len(modis_compat_files)}")
print(f"  PACE standard:    {len(pace_standard_files)}")
print(f"  PACE alt-sort:    {len(pace_altsort_files)}")

# Show by tile
for tile in TILES:
    tile_files = [f for f in tif_files if tile in f.name]
    print(f"\n{tile}: {len(tile_files)} files")


# =============================================================================
# Cell 9: Quick Visual Check - MODIS vs PACE Comparison
# =============================================================================

import matplotlib.pyplot as plt

def quick_visual_check_modis_vs_pace(tile: str = 'h08v04'):
    """Display MODIS-compatible metrics side-by-side."""
    
    fig, axes = plt.subplots(len(MODIS_METRICS), 2, figsize=(14, 5*len(MODIS_METRICS)))
    fig.suptitle(f'PACE vs MODIS: {tile}', fontsize=16, fontweight='bold')
    
    for i, metric in enumerate(MODIS_METRICS):
        # PACE
        pace_file = OUTPUT_DIR / f"PACE_{PACE_YEAR}_{tile}_{metric}.tif"
        if pace_file.exists():
            ds = gdal.Open(str(pace_file))
            pace_data = ds.GetRasterBand(1).ReadAsArray().astype(float)
            pace_data[pace_data == NO_DATA] = np.nan
            ds = None
            
            im = axes[i, 0].imshow(pace_data, cmap='viridis')
            axes[i, 0].set_title(f'PACE {PACE_YEAR}')
            axes[i, 0].set_ylabel(metric.replace('-', '\n'), fontsize=9)
            plt.colorbar(im, ax=axes[i, 0], shrink=0.8)
        else:
            axes[i, 0].text(0.5, 0.5, 'File not found', ha='center', va='center')
        
        # MODIS
        modis_file = OUTPUT_DIR / f"MODIS_{MODIS_YEAR}_{tile}_{metric}.tif"
        if modis_file.exists():
            ds = gdal.Open(str(modis_file))
            modis_data = ds.GetRasterBand(1).ReadAsArray().astype(float)
            modis_data[modis_data == NO_DATA] = np.nan
            ds = None
            
            im = axes[i, 1].imshow(modis_data, cmap='viridis')
            axes[i, 1].set_title(f'MODIS {MODIS_YEAR}')
            plt.colorbar(im, ax=axes[i, 1], shrink=0.8)
        else:
            axes[i, 1].text(0.5, 0.5, 'File not found', ha='center', va='center')
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / f"comparison_modis_vs_pace_{tile}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {fig_path}")
    plt.show()


# =============================================================================
# Cell 10: Visual Check - PACE Alternative Sorting Metrics
# =============================================================================

def quick_visual_check_altsort(tile: str = 'h08v04'):
    """Display PACE alternative sorting metrics."""
    
    # Select a subset of interesting alt-sort metrics to display
    display_metrics = [
        # Greenest vs Brownest
        ('PACEIndex-EVI-Brownest3Mean', 'EVI at Brownest\n(dormant/senescent)'),
        
        # Stress dynamics
        ('PACEIndex-EVI-LeastStressed3Mean', 'EVI at Least Stressed\n(high PRI)'),
        ('PACEIndex-EVI-MostStressed3Mean', 'EVI at Most Stressed\n(low PRI)'),
        
        # Anthocyanin dynamics
        ('PACEIndex-EVI-HighAnthocyanin3Mean', 'EVI at High Anthocyanin\n(stress/senescence)'),
        
        # Pigment dynamics
        ('PACEIndex-EVI-HighChlorophyll3Mean', 'EVI at High Chlorophyll\n(peak growth)'),
        ('PACEIndex-EVI-HighCarotenoid3Mean', 'EVI at High Carotenoid\n(senescence)'),
        
        # Phenology
        ('Phenology-MaxGreenupRate', 'Max Green-up Rate'),
        ('Phenology-GrowingSeasonLength', 'Growing Season Length\n(composites above median)'),
        
        # Cross-index
        ('CrossIndex-NDWI-DrySeason', 'NDWI Dry Season\n(water stress)'),
        ('CrossIndex-mARI-SeasonalRange', 'mARI Seasonal Range\n(anthocyanin variability)'),
    ]
    
    n_metrics = len(display_metrics)
    n_cols = 2
    n_rows = (n_metrics + 1) // 2
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4*n_rows))
    axes = axes.flatten()
    
    fig.suptitle(f'PACE Alternative Sorting Metrics: {tile}', fontsize=16, fontweight='bold')
    
    for i, (metric, label) in enumerate(display_metrics):
        pace_file = OUTPUT_DIR / f"PACE_{PACE_YEAR}_{tile}_{metric}.tif"
        
        if pace_file.exists():
            ds = gdal.Open(str(pace_file))
            data = ds.GetRasterBand(1).ReadAsArray().astype(float)
            data[data == NO_DATA] = np.nan
            ds = None
            
            # Choose colormap based on metric type
            if 'Stress' in metric or 'Brownest' in metric or 'Carotenoid' in metric:
                cmap = 'YlOrRd'  # Yellow-Orange-Red for stress/dormant
            elif 'Chlorophyll' in metric or 'Greenup' in metric:
                cmap = 'YlGn'   # Yellow-Green for growth
            elif 'NDWI' in metric or 'Water' in metric:
                cmap = 'Blues'  # Blue for water
            elif 'Anthocyanin' in metric or 'mARI' in metric:
                cmap = 'RdPu'   # Red-Purple for anthocyanins
            else:
                cmap = 'viridis'
            
            im = axes[i].imshow(data, cmap=cmap)
            axes[i].set_title(label, fontsize=10)
            plt.colorbar(im, ax=axes[i], shrink=0.8)
        else:
            axes[i].text(0.5, 0.5, f'{metric}\nNot found', ha='center', va='center', fontsize=8)
            axes[i].set_title(label, fontsize=10)
    
    # Hide any unused axes
    for i in range(n_metrics, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / f"comparison_altsort_{tile}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {fig_path}")
    plt.show()


# =============================================================================
# Cell 11: Visual Check - Phenology Metrics
# =============================================================================

def quick_visual_check_phenology(tile: str = 'h08v04'):
    """Display phenology-specific metrics."""
    
    phenology_metrics = [
        ('Phenology-MaxGreenupRate', 'Max Green-up Rate', 'Greens'),
        ('Phenology-MaxSenescenceRate', 'Max Senescence Rate', 'Oranges_r'),
        ('Phenology-GrowingSeasonLength', 'Growing Season Length', 'YlGn'),
        ('Phenology-TimeToPeak', 'Time to Peak (composite #)', 'plasma'),
        ('Phenology-TimeToMin', 'Time to Minimum (composite #)', 'plasma'),
        ('Phenology-SeasonalAsymmetry', 'Seasonal Asymmetry', 'RdBu'),
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    fig.suptitle(f'PACE Phenology Metrics: {tile}', fontsize=16, fontweight='bold')
    
    for i, (metric, label, cmap) in enumerate(phenology_metrics):
        pace_file = OUTPUT_DIR / f"PACE_{PACE_YEAR}_{tile}_{metric}.tif"
        
        if pace_file.exists():
            ds = gdal.Open(str(pace_file))
            data = ds.GetRasterBand(1).ReadAsArray().astype(float)
            data[data == NO_DATA] = np.nan
            ds = None
            
            im = axes[i].imshow(data, cmap=cmap)
            axes[i].set_title(label, fontsize=12)
            plt.colorbar(im, ax=axes[i], shrink=0.8)
        else:
            axes[i].text(0.5, 0.5, f'Not found:\n{metric}', ha='center', va='center')
            axes[i].set_title(label, fontsize=12)
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / f"comparison_phenology_{tile}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {fig_path}")
    plt.show()


# =============================================================================
# Cell 12: Run All Visualizations
# =============================================================================

print("="*70)
print("GENERATING COMPARISON FIGURES")
print("="*70)

for tile in TILES:
    print(f"\n--- {tile} ---")
    
    print("  MODIS vs PACE comparison...")
    quick_visual_check_modis_vs_pace(tile)
    
    print("  Alternative sorting metrics...")
    quick_visual_check_altsort(tile)
    
    print("  Phenology metrics...")
    quick_visual_check_phenology(tile)

print("\n" + "="*70)
print("DONE! All comparison figures saved.")
print("="*70)
